# Merge the Patch 16.19 Final Dataset

This notebook merges the Riot OTP observations with the Global D2+ LoLalytics matchup table. It preserves every Riot row, flags unavailable and low-sample matchups, checks that the merge creates no duplicate observations, and exports the combined dataset.

In [1]:
%pip install -q pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd

TARGET_PATCH = '16.19'
TARGET_RANK = 'd2_plus'
TARGET_REGION = 'global'
RECOMMENDED_MIN_GAMES = 50  # Flag only; no rows are deleted from the full export

start_dir = Path.cwd().resolve()
search_dirs = [start_dir, *start_dir.parents]
PROJECT_DIR = next(
    (p for p in search_dirs if (p / 'data' / 'processed').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not find the repository data/processed directory.')

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
print('Project directory:', PROJECT_DIR)

Project directory: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction


In [3]:
def load_preferred(stem):
    parquet_path = PROCESSED_DIR / f'{stem}.parquet'
    csv_path = PROCESSED_DIR / f'{stem}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'Could not find {parquet_path} or {csv_path}.')

otp = load_preferred('otp_matches_16_19')
matchups = load_preferred('expected_matchups_16_19')

print(f'OTP observations: {len(otp):,}')
print(f'Expected-matchup rows: {len(matchups):,}')
display(matchups['status'].value_counts(dropna=False).rename('rows').to_frame())

OTP observations: 13,650
Expected-matchup rows: 7,043


,rows
status,
ok,6886
error,157


## Prepare a one-row-per-matchup lookup

In [4]:
KEY = ['lane', 'champion', 'opponent_champion']
required_otp = {'match_id', 'puuid', *KEY}
required_matchups = {
    'patch', 'rank', 'region', 'status', 'expected_winrate', 'matchup_games', *KEY
}
missing_otp = required_otp.difference(otp.columns)
missing_matchups = required_matchups.difference(matchups.columns)
if missing_otp:
    raise ValueError(f'OTP table is missing columns: {sorted(missing_otp)}')
if missing_matchups:
    raise ValueError(f'Matchup table is missing columns: {sorted(missing_matchups)}')

lookup = matchups.copy()
lookup['patch'] = lookup['patch'].astype(str)
lookup = lookup.loc[
    lookup['patch'].eq(TARGET_PATCH)
    & lookup['rank'].eq(TARGET_RANK)
    & lookup['region'].eq(TARGET_REGION)
].copy()
lookup['expected_winrate'] = pd.to_numeric(lookup['expected_winrate'], errors='coerce')
lookup['matchup_games'] = pd.to_numeric(lookup['matchup_games'], errors='coerce')

if lookup.duplicated(KEY).any():
    duplicates = lookup.loc[lookup.duplicated(KEY, keep=False), KEY + ['status']]
    raise ValueError(f'Duplicate matchup keys found:\n{duplicates.head(20).to_string(index=False)}')

lookup = lookup.rename(columns={
    'rank': 'matchup_rank',
    'region': 'matchup_region',
    'status': 'matchup_status',
    'patch': 'matchup_patch',
})
keep_optional = [
    'raw_matchup_winrate', 'tier_average_winrate', 'lolalytics_delta2',
    'source_url', 'error', 'scraped_at_utc'
]
lookup_columns = KEY + [
    'matchup_patch', 'matchup_rank', 'matchup_region', 'matchup_status',
    'expected_winrate', 'matchup_games'
] + [column for column in keep_optional if column in lookup.columns]
lookup = lookup.loc[:, lookup_columns]
print(f'Validated lookup rows: {len(lookup):,}')

Validated lookup rows: 7,043


## Merge without dropping observations

In [5]:
original_rows = len(otp)
merged = otp.merge(lookup, on=KEY, how='left', validate='many_to_one')

if len(merged) != original_rows:
    raise AssertionError('The merge changed the number of OTP observations.')
if merged.duplicated(['match_id', 'puuid']).any():
    raise AssertionError('The merge created duplicate seed-player match rows.')

merged['expected_winrate_prob'] = merged['expected_winrate'] / 100
merged['matchup_data_available'] = merged['matchup_status'].eq('ok')
for threshold in (10, 25, 50, 100, 200):
    merged[f'matchup_games_ge_{threshold}'] = (
        merged['matchup_data_available'] & merged['matchup_games'].ge(threshold)
    )
merged['matchup_reliable'] = merged[f'matchup_games_ge_{RECOMMENDED_MIN_GAMES}']

print(f'Merged observations: {len(merged):,}')
print(f'Unique Riot matches: {merged["match_id"].nunique():,}')
print(f'Rows with successful matchup data: {merged["matchup_data_available"].sum():,} '
      f'({merged["matchup_data_available"].mean():.1%})')

Merged observations: 13,650
Unique Riot matches: 2,776
Rows with successful matchup data: 13,490 (98.8%)


## Compare possible minimum-game cutoffs

This reports how many player observations and unique Riot matches each cutoff would retain. The full export remains unfiltered.

In [ ]:
summary_rows = []
for threshold in (0, 10, 25, 50, 100, 200):
    mask = merged["matchup_data_available"].copy()
    if threshold:
        mask &= merged['matchup_games'].ge(threshold)
    summary_rows.append({
        'minimum_matchup_games': threshold,
        'player_rows_retained': int(mask.sum()),
        'player_rows_retained_pct': float(mask.mean()),
        'unique_matches_retained': int(merged.loc[mask, 'match_id'].nunique()),
        'unique_matchup_keys_retained': int(merged.loc[mask, KEY].drop_duplicates().shape[0]),
    })
quality_summary = pd.DataFrame(summary_rows)
display(quality_summary.style.format({'player_rows_retained_pct': '{:.1%}'}))

unresolved = (
    merged.loc[~merged['matchup_data_available'], KEY + ['matchup_status']]
    .drop_duplicates()
    .sort_values(KEY)
    .reset_index(drop=True)
)
print(f'Unresolved matchup keys represented in the OTP data: {len(unresolved):,}')
display(unresolved.head(20))

,minimum_matchup_games,player_rows_retained,player_rows_retained_pct,unique_matches_retained,unique_matchup_keys_retained
0,0,13490,98.8%,2774,6886
1,10,12599,92.3%,2749,6060
2,25,11858,86.9%,2736,5388
3,50,11001,80.6%,2701,4685
4,100,9726,71.3%,2625,3720
5,200,7550,55.3%,2461,2383


Unresolved matchup keys represented in the OTP data: 4,660


,lane,champion,opponent_champion,matchup_status
0,adc,Ahri,Viktor,ok
1,adc,Akshan,Caitlyn,ok
2,adc,Akshan,Kaisa,ok
3,adc,Akshan,KogMaw,ok
4,adc,Akshan,Tristana,ok
5,adc,Ambessa,Caitlyn,ok
6,adc,Ambessa,KogMaw,ok
7,adc,Anivia,Jhin,ok
8,adc,Anivia,Viktor,ok
9,adc,Annie,Jhin,ok


## Save the combined dataset

In [7]:
output_stem = 'otp_analysis_16_19'
csv_path = PROCESSED_DIR / f'{output_stem}.csv'
parquet_path = PROCESSED_DIR / f'{output_stem}.parquet'
summary_path = PROCESSED_DIR / 'matchup_quality_summary_16_19.csv'
unresolved_path = PROCESSED_DIR / 'unresolved_matchups_16_19.csv'

merged.to_csv(csv_path, index=False)
merged.to_parquet(parquet_path, index=False)
quality_summary.to_csv(summary_path, index=False)
unresolved.to_csv(unresolved_path, index=False)

print('Final CSV:', csv_path)
print('Final Parquet:', parquet_path)
print('Cutoff comparison:', summary_path)
print('Unresolved matchups:', unresolved_path)

Final CSV: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/data/processed/otp_analysis_16_19.csv
Final Parquet: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/data/processed/otp_analysis_16_19.parquet
Cutoff comparison: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/data/processed/matchup_quality_summary_16_19.csv
Unresolved matchups: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/data/processed/unresolved_matchups_16_19.csv


In [8]:
model_df = merged.loc[merged["matchup_games_ge_50"]].copy()

model_df.to_csv(
    PROCESSED_DIR / "otp_analysis_16_19_min50.csv",
    index=False,
)

model_df.to_parquet(
    PROCESSED_DIR / "otp_analysis_16_19_min50.parquet",
    index=False,
)

print(model_df.shape)

(11001, 35)
